# Financial Sentiment Project Walkthrough

This notebook is a readable version of the project pipeline. It does **not** reimplement the project from scratch. Instead, it reuses the real functions and saved artifacts from `src/` and `outputs/` so that the project can be understood in one place.

Main goals:
- inspect each dataset clearly in `pandas`
- show how labels and splits were built
- summarize what each model did
- load the saved evaluation outputs and explain what they mean
- keep the logic aligned with the actual `.py` pipeline

In [2]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, Image, display

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import FIGURES_DIR, METRICS_DIR, SUMMARIES_DIR
from src.data_pipeline import (
    load_fiqa_all,
    load_lm_restricted_vocabulary,
    load_nosible,
    load_nosible_splits,
    load_phrasebank,
    load_phrasebank_splits,
)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 20)
plt.style.use("default")

PROJECT_ROOT

ModuleNotFoundError: No module named 'huggingface_hub'

## 1. What the project does

The project is a **financial sentiment classification** project. The main supervised setup is:

- train on **Financial PhraseBank**
- validate and test on PhraseBank splits
- also check transfer performance on **FiQA**
- compare several model families:
  - classical bag-of-words logistic regression
  - Loughran-McDonald restricted bag-of-words
  - a GRU neural model
  - raw FinBERT
  - fine-tuned FinBERT

There is also a second experiment using **NOSIBLE**, which is larger and useful as an extension, but the core class project story is the PhraseBank + FiQA pipeline.

In [ ]:
phrasebank = load_phrasebank()
phrasebank_splits = load_phrasebank_splits()
fiqa = load_fiqa_all()
nosible = load_nosible()
nosible_splits = load_nosible_splits()
lm_vocab = load_lm_restricted_vocabulary()

summary_rows = [
    {"dataset": "Financial PhraseBank (full)", "rows": len(phrasebank), "labels": phrasebank['label'].value_counts().to_dict()},
    {"dataset": "PhraseBank train", "rows": len(phrasebank_splits['train']), "labels": phrasebank_splits['train']['label'].value_counts().to_dict()},
    {"dataset": "PhraseBank validation", "rows": len(phrasebank_splits['validation']), "labels": phrasebank_splits['validation']['label'].value_counts().to_dict()},
    {"dataset": "PhraseBank test", "rows": len(phrasebank_splits['test']), "labels": phrasebank_splits['test']['label'].value_counts().to_dict()},
    {"dataset": "FiQA train", "rows": len(fiqa['train']), "labels": fiqa['train']['label'].value_counts().to_dict()},
    {"dataset": "FiQA valid", "rows": len(fiqa['valid']), "labels": fiqa['valid']['label'].value_counts().to_dict()},
    {"dataset": "FiQA test", "rows": len(fiqa['test']), "labels": fiqa['test']['label'].value_counts().to_dict()},
    {"dataset": "NOSIBLE (full)", "rows": len(nosible), "labels": nosible['label'].value_counts().to_dict()},
]

pd.DataFrame(summary_rows)

The table above is the quickest way to understand the data universe.

- **PhraseBank** is the main labeled benchmark dataset used for training and in-domain evaluation.
- **FiQA** is smaller and more difficult; it is used more like a transfer / robustness test.
- **NOSIBLE** is a large additional dataset used in the extension experiment.
- The Loughran-McDonald dictionary is not a dataset of examples; it is a **financial vocabulary restriction** used by one of the classical baselines.

## 2. Inspect the raw data clearly

Below, each block shows a few real rows so the structure of each dataset is easy to understand.

In [ ]:
display(Markdown("### Financial PhraseBank sample"))
display(phrasebank[['source_index', 'text', 'normalized_text', 'label', 'label_id']].head(8))

display(Markdown("### FiQA sample"))
display(fiqa['train'][['text', 'score', 'label', 'label_id']].head(8))

display(Markdown("### NOSIBLE sample"))
display(nosible[['source_index', 'text', 'normalized_text', 'label', 'label_id']].head(8))

display(Markdown("### Loughran-McDonald restricted vocabulary sample"))
pd.DataFrame({'lm_token_sample': sorted(list(lm_vocab))[:25]})

What was done here:

- Each sentence dataset is converted into a standardized table with:
  - `text`
  - `normalized_text`
  - `label`
  - `label_id`
- PhraseBank already has categorical labels.
- FiQA starts from a continuous score and is mapped to `negative / neutral / positive` using thresholds.
- NOSIBLE is already a sentence-level sentiment dataset with labels.
- The Loughran-McDonald lexicon is turned into a **restricted token set**, which is later used in one of the bag-of-words baselines.

## 3. Phase 1 preprocessing and split logic

In [ ]:
phase1_summary = json.loads((SUMMARIES_DIR / 'phase1_summary.json').read_text(encoding='utf-8'))
nosible_summary = json.loads((SUMMARIES_DIR / 'nosible_summary.json').read_text(encoding='utf-8'))

display(Markdown("### Saved PhraseBank / FiQA preprocessing summary"))
display(pd.json_normalize(phase1_summary, sep=' -> ').T)

display(Markdown("### Saved NOSIBLE preprocessing summary"))
display(pd.json_normalize(nosible_summary, sep=' -> ').T.head(20))

What Phase 1 did:

- downloaded / loaded the raw datasets
- normalized the text
- mapped labels into the shared 3-class setup
- created reproducible stratified train / validation / test splits
- saved split indices as artifacts
- ran smoke checks to confirm row counts and label validity

For the main project, the important split is the **PhraseBank train / validation / test** split. That is the training backbone for the supervised comparison.

In [ ]:
split_preview = {
    'PhraseBank train': phrasebank_splits['train'][['source_index', 'text', 'label']].head(5),
    'PhraseBank validation': phrasebank_splits['validation'][['source_index', 'text', 'label']].head(5),
    'PhraseBank test': phrasebank_splits['test'][['source_index', 'text', 'label']].head(5),
}

for name, frame in split_preview.items():
    display(Markdown(f"### {name}"))
    display(frame)

These previews make the split idea concrete: the project is not just a black box. It is a conventional supervised workflow where the sentences are split once, saved, and reused by all downstream models.

## 4. Classical baselines

In [ ]:
classical_training = json.loads((METRICS_DIR / 'classical_training_summary.json').read_text(encoding='utf-8'))
evaluation = pd.read_csv(METRICS_DIR / 'evaluation_summary.csv')

display(Markdown("### Classical training summary"))
display(pd.DataFrame(classical_training).T)

display(Markdown("### Main evaluation table"))
display(evaluation.sort_values(['split', 'macro_f1'], ascending=[True, False]))

What was done here:

- **`standard_bow`**: TF-IDF bag-of-words + logistic regression baseline
- **`lm_restricted_bow`**: the same general idea, but restricted to the Loughran-McDonald finance lexicon
- hyperparameter `C` was tuned on the validation split

How to read the current saved results:

- `standard_bow` is the stronger classical baseline
- `lm_restricted_bow` is more interpretable from a finance-dictionary perspective, but it loses a lot of predictive power
- both classical models degrade sharply on **FiQA**, which shows domain transfer is hard

In [ ]:
main_pivot = evaluation.pivot(index='model', columns='split', values='macro_f1').round(3)
main_pivot

This pivot is one of the most important project tables.

In the saved outputs:
- `standard_bow` gets around **0.706 macro-F1** on PhraseBank test
- `lm_restricted_bow` gets around **0.463 macro-F1** on PhraseBank test

So the finance lexicon alone is not enough. It is a useful benchmark, but not the best-performing pipeline.

## 5. Neural and transformer models

In [ ]:
gru_training = json.loads((METRICS_DIR / 'gru_training_summary.json').read_text(encoding='utf-8'))
finetuned_training = json.loads((METRICS_DIR / 'finbert_finetuned_training_summary.json').read_text(encoding='utf-8'))

gru_history = pd.DataFrame(gru_training['history'])
finetuned_history = pd.DataFrame(finetuned_training['history'])

display(Markdown('### GRU training history'))
display(gru_history)

display(Markdown('### Fine-tuned FinBERT training history'))
display(finetuned_history)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(gru_history['epoch'], gru_history['validation_accuracy'], marker='o', label='GRU validation accuracy')
axes[0].plot(finetuned_history['epoch'], finetuned_history['validation_accuracy'], marker='o', label='FinBERT validation accuracy')
axes[0].set_title('Validation accuracy by epoch')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(gru_history['epoch'], gru_history['validation_macro_f1'], marker='o', label='GRU validation macro-F1')
axes[1].plot(finetuned_history['epoch'], finetuned_history['validation_macro_f1'], marker='o', label='FinBERT validation macro-F1')
axes[1].set_title('Validation macro-F1 by epoch')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

What was done here:

- **GRU**: a learned sequence model trained from scratch on PhraseBank
- **raw FinBERT**: a pretrained finance-specific transformer used directly
- **fine-tuned FinBERT**: the same pretrained model, adapted to the PhraseBank labels

How to interpret the saved results:

- the GRU is better than the weak lexicon baseline, but it does **not** beat FinBERT
- raw FinBERT is already very strong
- fine-tuned FinBERT is the best in-domain model on PhraseBank
- transfer to FiQA is mixed: raw FinBERT actually beats fine-tuned FinBERT there in the saved outputs

In [ ]:
best_by_split = evaluation.sort_values(['split', 'macro_f1'], ascending=[True, False]).groupby('split').head(3)
best_by_split[['split', 'model', 'accuracy', 'macro_f1']].reset_index(drop=True)

Current headline results from the saved outputs:

- **PhraseBank validation**: fine-tuned FinBERT is best
- **PhraseBank test**: fine-tuned FinBERT is best
- **FiQA test**: raw FinBERT is best

That tells a coherent story:
- fine-tuning helps on the in-domain benchmark
- but raw pretrained finance language understanding generalizes a bit better to the out-of-domain FiQA test set

## 6. Confusion matrices

In [ ]:
image_paths = [
    FIGURES_DIR / 'standard_bow_phrasebank_test_confusion_matrix.png',
    FIGURES_DIR / 'gru_phrasebank_test_confusion_matrix.png',
    FIGURES_DIR / 'raw_finbert_phrasebank_test_confusion_matrix.png',
    FIGURES_DIR / 'finbert_finetuned_phrasebank_test_confusion_matrix.png',
]

for path in image_paths:
    display(Markdown(f"### {path.name}"))
    display(Image(filename=str(path)))

Why show confusion matrices?

Accuracy alone can hide class imbalance effects. These figures show where each model is making mistakes. In sentiment projects, the usual weak point is the **neutral** class or the confusion between mild positive / mild neutral language. The confusion matrices make that visible.

## 7. NOSIBLE extension experiment

In [ ]:
nosible_eval = pd.read_csv(METRICS_DIR / 'nosible_evaluation_summary.csv')
nosible_eval.sort_values(['split', 'macro_f1'], ascending=[True, False])

This extension asks a slightly different question: what happens if the models are trained on a much larger sentiment dataset and then evaluated across datasets?

Saved result summary:
- on **NOSIBLE itself**, the standard bag-of-words baseline is very competitive
- transfer back to PhraseBank and FiQA is still limited
- so bigger data alone does not automatically solve cross-dataset generalization

## 8. Final interpretation

This project is strongest when presented as a **clean supervised sentiment benchmark in finance**:

- build a shared 3-class label setup
- compare classical NLP, a neural RNN, and finance-specific transformers
- evaluate both in-domain and out-of-domain

The saved outputs support these conclusions:

- finance-specific transformers are the strongest models overall
- fine-tuned FinBERT is best on PhraseBank
- raw FinBERT is best on FiQA in the current saved outputs
- classical lexicon restriction is interpretable, but too weak to be the headline method
- cross-dataset transfer remains difficult, which is an honest and useful result

That makes the project academically clean: the notebook now shows the actual data, the actual splits, the actual training summaries, and the actual evaluation outputs in one place.